In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt

# 加载数据
with h5py.File('/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED/train.h5', 'r') as f:
    X = f['X'][()]
    y = f['y'][()]

# 画第一个样本的前3个通道
plt.figure(figsize=(12, 6))
for i in range(3):
    plt.subplot(3, 1, i+1)
    plt.plot(X[0, i, :])
    plt.title(f'Channel {i}, Label={y[0]}')
    plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

# 打印基本统计
print(f"X shape: {X.shape}")
print(f"X min: {X.min():.4f}, max: {X.max():.4f}, mean: {X.mean():.4f}, std: {X.std():.4f}")
print(f"Label distribution: {np.bincount(y)}")

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.signal import welch

# 加载
with h5py.File('/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED/train.h5', 'r') as f:
    X = f['X'][()]
    y = f['y'][()]

# 选一个"正面情绪"和"负面情绪"的样本
pos_idx = np.where(y == 2)[0][0]   # positive
neg_idx = np.where(y == 0)[0][0]   # negative

# 只看通道0，用 Welch 方法看功率谱
fs = 200
f_pos, P_pos = welch(X[pos_idx, 0, :], fs=fs, nperseg=128)
f_neg, P_neg = welch(X[neg_idx, 0, :], fs=fs, nperseg=128)

plt.figure(figsize=(10, 5))
plt.semilogy(f_pos, P_pos, label=f'Positive (label=2)', alpha=0.7)
plt.semilogy(f_neg, P_neg, label=f'Negative (label=0)', alpha=0.7)
plt.axvline(8, color='gray', linestyle='--', alpha=0.3)
plt.axvline(14, color='gray', linestyle='--', alpha=0.3)
plt.axvline(31, color='gray', linestyle='--', alpha=0.3)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power')
plt.title('Channel 0: Power Spectrum by Emotion')
plt.legend()
plt.xlim(0, 50)
plt.show()

# 再看一下：不同标签的样本，alpha波段(8-14Hz)平均功率
alpha_power = []
for label in [0, 1, 2]:
    idx = np.where(y == label)[0]
    powers = []
    for i in idx[:50]:  # 只看前50个，省时间
        f, P = welch(X[i, 0, :], fs=fs, nperseg=128)
        alpha_mask = (f >= 8) & (f <= 14)
        powers.append(np.mean(P[alpha_mask]))
    alpha_power.append(np.mean(powers))
    print(f"Label {label}: Alpha power = {np.mean(powers):.4f} ± {np.std(powers):.4f}")

print(f"\nAlpha power by label: {alpha_power}")

In [ ]:
"""
SEED Dataset: EEGNet - 无标准化版本
=====================================
不做任何z-score标准化，直接用原始数据训练。
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 100
LR = 0.001
PATIENCE = 15

print(f"使用设备: {DEVICE}")

# =================== 数据加载（无标准化） ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print("[步骤2] 无标准化，直接转为Tensor...")
# 直接转为Tensor，不做任何标准化
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

# =================== EEGNet模型 ===================

class EEGNet(nn.Module):
    def __init__(self, num_classes=3, channels=62, samples=400, 
                 dropout_rate=0.5, kernel_length=64, F1=8, D=2):
        super(EEGNet, self).__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 16), padding=(0, 8), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        self.flatten_dim = self.F2 * 12
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# =================== 训练函数 ===================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主训练流程 ===================

print("\n[步骤3] 构建EEGNet模型...")
model = EEGNet(num_classes=3, channels=62, samples=400, 
               dropout_rate=0.5, kernel_length=64, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤4] 开始训练 (无标准化)...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = model.state_dict().copy()
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 最终评估 ===================

print(f"\n[步骤5] 最终评估...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['negative', 'neutral', 'positive']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤6] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'seed_eegnet_noscale_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   negative={np.sum(test_pred==0)}, neutral={np.sum(test_pred==1)}, positive={np.sum(test_pred==2)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

使用设备: cuda
[步骤1] 加载数据...
[步骤2] 无标准化，直接转为Tensor...

[步骤3] 构建EEGNet模型...
模型总参数量: 2,675

[步骤4] 开始训练 (无标准化)...
------------------------------------------------------------
Epoch   1/100 | Train: 0.3511 | Val: 0.3289
Epoch   5/100 | Train: 0.3789 | Val: 0.3622
Epoch  10/100 | Train: 0.4567 | Val: 0.4000
Epoch  15/100 | Train: 0.5022 | Val: 0.3911
Epoch  20/100 | Train: 0.5156 | Val: 0.4200
Epoch  25/100 | Train: 0.5433 | Val: 0.4133
Epoch 00029: reducing learning rate of group 0 to 5.0000e-04.
Epoch  30/100 | Train: 0.5556 | Val: 0.4267
Epoch 00035: reducing learning rate of group 0 to 2.5000e-04.
Epoch  35/100 | Train: 0.5822 | Val: 0.4000

⏹️ Early stopping (best=0.4422 at epoch 23)

[步骤5] 最终评估...
------------------------------------------------------------
验证集准确率: 0.4178
              precision    recall  f1-score   support

    negative       0.48      0.65      0.55       150
     neutral       0.34      0.27      0.30       150
    positive       0.39      0.33      0.36       150

  

In [5]:
"""
SEED Dataset: EEGNet 小模型 + 数据增强 + 标签平滑
====================================================
用小EEGNet（2,675参数），加两个防过拟合技巧：
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 30
NOISE_STD = 0.01  # 噪声强度（很小）
LABEL_SMOOTHING = 0.1  # 标签平滑

print(f"使用设备: {DEVICE}")

# =================== 数据加载（无标准化） ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

X_train = train_data['X']
y_train = train_data['y']
X_val = val_data['X']
y_val = val_data['y']
X_test = test_data['X']

print("[步骤2] 无标准化，直接转为Tensor...")
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

# =================== EEGNet（小模型） ===================

class EEGNet(nn.Module):
    def __init__(self, num_classes=3, channels=62, samples=400, 
                 dropout_rate=0.5, kernel_length=64, F1=8, D=2):
        super(EEGNet, self).__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 16), padding=(0, 8), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        self.flatten_dim = self.F2 * 12
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# =================== 训练函数（带数据增强） ===================

def add_noise(x, std=0.01):
    """添加高斯噪声"""
    noise = torch.randn_like(x) * std
    return x + noise

def train_epoch(model, loader, optimizer, criterion, device, noise_std=0.01):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        batch_x = add_noise(batch_x, noise_std)  # 加噪声！
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主训练流程 ===================

print("\n[步骤3] 构建 EEGNet (小模型 + 增强)...")
model = EEGNet(num_classes=3, channels=62, samples=400, 
               dropout_rate=0.5, kernel_length=64, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

# 标签平滑 + 更强的L2
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=5e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤4] 开始训练...")
print(f"  - 噪声增强: std={NOISE_STD}")
print(f"  - 标签平滑: {LABEL_SMOOTHING}")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE, noise_std=NOISE_STD)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = model.state_dict().copy()
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 最终评估 ===================

print(f"\n[步骤5] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['negative', 'neutral', 'positive']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤6] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'seed_eegnet_aug_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   negative={np.sum(test_pred==0)}, neutral={np.sum(test_pred==1)}, positive={np.sum(test_pred==2)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

使用设备: cuda
[步骤1] 加载数据...
[步骤2] 无标准化，直接转为Tensor...

[步骤3] 构建 EEGNet (小模型 + 增强)...
模型总参数量: 2,675

[步骤4] 开始训练...
  - 噪声增强: std=0.01
  - 标签平滑: 0.1
------------------------------------------------------------
Epoch   1/150 | Train: 0.3222 | Val: 0.3422
Epoch   5/150 | Train: 0.3978 | Val: 0.3733
Epoch  10/150 | Train: 0.4411 | Val: 0.3756
Epoch  15/150 | Train: 0.4856 | Val: 0.3867
Epoch  20/150 | Train: 0.5056 | Val: 0.4067
Epoch 00024: reducing learning rate of group 0 to 5.0000e-04.
Epoch  25/150 | Train: 0.5378 | Val: 0.4156
Epoch  30/150 | Train: 0.5300 | Val: 0.3889
Epoch  35/150 | Train: 0.5400 | Val: 0.3822
Epoch 00036: reducing learning rate of group 0 to 2.5000e-04.
Epoch  40/150 | Train: 0.5611 | Val: 0.4044
Epoch  45/150 | Train: 0.5722 | Val: 0.3956
Epoch 00047: reducing learning rate of group 0 to 1.2500e-04.
Epoch  50/150 | Train: 0.5856 | Val: 0.3911
Epoch  55/150 | Train: 0.6100 | Val: 0.4067

⏹️ Early stopping (best=0.4156 at epoch 25)

[步骤5] 最终评估 (Epoch 25)...
-----------

In [1]:
"""
SEED Dataset: EEGNet + 90%训练 + 10%验证
========================================
合并train+val后，再分出10%作为验证集监控。
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
EPOCHS = 150
LR = 0.001
PATIENCE = 25
VAL_SPLIT = 0.1  # 留10%做验证

print(f"使用设备: {DEVICE}")

# =================== 数据加载 ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

# 合并 train + val
X_combined = np.concatenate([train_data['X'], val_data['X']], axis=0)
y_combined = np.concatenate([train_data['y'], val_data['y']], axis=0)
X_test = test_data['X']

print(f"  合并后总数据: {X_combined.shape}")

# 分出10%做验证（随机分，保持标签比例）
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y_combined, test_size=VAL_SPLIT, 
    random_state=42, stratify=y_combined
)

print(f"  训练集: {X_train.shape}")
print(f"  验证集: {X_val.shape}")
print(f"  测试集: {X_test.shape}")

print("[步骤2] 转为Tensor...")
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

# =================== EEGNet ===================

class EEGNet(nn.Module):
    def __init__(self, num_classes=3, channels=62, samples=400, 
                 dropout_rate=0.5, kernel_length=64, F1=8, D=2):
        super(EEGNet, self).__init__()
        self.F1 = F1
        self.D = D
        self.F2 = F1 * D
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length//2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, self.F2, (channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout_rate)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(self.F2, self.F2, (1, 16), padding=(0, 8), groups=self.F2, bias=False),
            nn.Conv2d(self.F2, self.F2, 1, bias=False),
            nn.BatchNorm2d(self.F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout_rate)
        )
        
        self.flatten_dim = self.F2 * 12
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_dim, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.classifier(x)
        return x

# =================== 训练函数 ===================

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 主训练流程 ===================

print("\n[步骤3] 构建 EEGNet...")
model = EEGNet(num_classes=3, channels=62, samples=400, 
               dropout_rate=0.5, kernel_length=64, F1=8, D=2).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型总参数量: {total_params:,}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"\n[步骤4] 开始训练 ({EPOCHS}轮)...")
print("-" * 60)

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

model.load_state_dict(best_model_state)

# =================== 最终评估 ===================

print(f"\n[步骤5] 最终评估 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_epoch(model, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['negative', 'neutral', 'positive']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[步骤6] 生成测试集预测...")
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = model(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'seed_split_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n 已保存: {output_file}")
print(f"   negative={np.sum(test_pred==0)}, neutral={np.sum(test_pred==1)}, positive={np.sum(test_pred==2)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

使用设备: cuda
[步骤1] 加载数据...
  合并后总数据: (1350, 62, 400)
  训练集: (1215, 62, 400)
  验证集: (135, 62, 400)
  测试集: (450, 62, 400)
[步骤2] 转为Tensor...

[步骤3] 构建 EEGNet...
模型总参数量: 2,675

[步骤4] 开始训练 (150轮)...
------------------------------------------------------------
Epoch   1/150 | Train: 0.3514 | Val: 0.3333
Epoch   5/150 | Train: 0.4198 | Val: 0.4148
Epoch  10/150 | Train: 0.4823 | Val: 0.3852
Epoch  15/150 | Train: 0.4864 | Val: 0.4444
Epoch  20/150 | Train: 0.5144 | Val: 0.4889
Epoch 00023: reducing learning rate of group 0 to 5.0000e-04.
Epoch  25/150 | Train: 0.5399 | Val: 0.4963
Epoch  30/150 | Train: 0.5761 | Val: 0.4889
Epoch 00033: reducing learning rate of group 0 to 2.5000e-04.
Epoch  35/150 | Train: 0.5506 | Val: 0.4889
Epoch  40/150 | Train: 0.5885 | Val: 0.4667
Epoch 00042: reducing learning rate of group 0 to 1.2500e-04.
Epoch  45/150 | Train: 0.5679 | Val: 0.4963

 Early stopping (best=0.5037 at epoch 24)

[步骤5] 最终评估 (Epoch 24)...
----------------------------------------------------

In [2]:
"""
SEED Dataset: AutoEncoder 自监督预训练 + 分类
===============================================
在你们的EEG数据上预训练一个自编码器，
让模型先学会"EEG长什么样"，
然后再学分类。
"""

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# =================== 配置 ===================
DATA_DIR = '/mnt/dataset0/Jity/EEG-Conformer-main/EEG-Conformer-main/data/SEED'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = 32
PRETRAIN_EPOCHS = 80
FINETUNE_EPOCHS = 100
LR = 0.001
PATIENCE = 20

print(f"使用设备: {DEVICE}")

# =================== 数据加载 ===================

def load_h5(filepath):
    data_dict = {}
    with h5py.File(filepath, 'r') as f:
        for key in f.keys():
            data_dict[key] = f[key][()]
    return data_dict

print("[步骤1] 加载数据...")
train_data = load_h5(f'{DATA_DIR}/train.h5')
val_data = load_h5(f'{DATA_DIR}/val.h5')
test_data = load_h5(f'{DATA_DIR}/test_x_only.h5')

# 合并所有有标签的数据用于预训练
X_pretrain = np.concatenate([train_data['X'], val_data['X']], axis=0)
y_all = np.concatenate([train_data['y'], val_data['y']], axis=0)
X_test = test_data['X']

# 从合并数据里分出 train/val 用于分类
X_train, X_val, y_train, y_val = train_test_split(
    X_pretrain, y_all, test_size=0.15, random_state=42, stratify=y_all
)

print(f"  预训练数据: {X_pretrain.shape}")
print(f"  分类训练集: {X_train.shape}")
print(f"  分类验证集: {X_val.shape}")

# 转为Tensor
X_pretrain_t = torch.FloatTensor(X_pretrain).unsqueeze(1)
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train)
X_val_t = torch.FloatTensor(X_val).unsqueeze(1)
y_val_t = torch.LongTensor(y_val)
X_test_t = torch.FloatTensor(X_test).unsqueeze(1)

pretrain_loader = DataLoader(TensorDataset(X_pretrain_t, X_pretrain_t), 
                              batch_size=BATCH_SIZE, shuffle=True)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)

# =================== 自编码器 ===================

class EEGEncoder(nn.Module):
    """把EEG压缩成低维表示"""
    def __init__(self, channels=62, samples=400, latent_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, (1, 32), padding=(0, 16)),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.AvgPool2d((1, 2)),
            
            nn.Conv2d(16, 32, (channels, 1)),  # 跨通道压缩
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AvgPool2d((1, 2)),
            
            nn.Flatten(),
            nn.Linear(32 * 100, latent_dim)
        )
        
    def forward(self, x):
        return self.encoder(x)

class EEGDecoder(nn.Module):
    """从低维表示还原EEG"""
    def __init__(self, channels=62, samples=400, latent_dim=64):
        super().__init__()
        self.channels = channels
        self.samples = samples
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 100),
            nn.Unflatten(1, (32, 1, 100)),
            
            nn.ConvTranspose2d(32, 16, (1, 4), stride=(1, 2), padding=(0, 1)),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            
            nn.ConvTranspose2d(16, 1, (channels, 4), stride=(1, 2), padding=(0, 1)),
        )
        
    def forward(self, x):
        x = self.decoder(x)
        return x[:, :, :self.channels, :self.samples]

class EEGAutoEncoder(nn.Module):
    def __init__(self, channels=62, samples=400, latent_dim=64):
        super().__init__()
        self.encoder = EEGEncoder(channels, samples, latent_dim)
        self.decoder = EEGDecoder(channels, samples, latent_dim)
        
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon, z

# =================== 分类器 ===================

class EEGClassifier(nn.Module):
    """在预训练编码器上接分类头（编码器冻结）"""
    def __init__(self, encoder, num_classes=3, latent_dim=64):
        super().__init__()
        self.encoder = encoder
        # 冻结编码器！
        for param in self.encoder.parameters():
            param.requires_grad = False
            
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
        
    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z)

# =================== 训练函数 ===================

def pretrain_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch_x, _ in loader:
        batch_x = batch_x.to(device)
        optimizer.zero_grad()
        recon, z = model(batch_x)
        loss = nn.MSELoss()(recon, batch_x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def train_clf_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
    return total_loss / len(loader), correct / total

def eval_clf_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            total_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += batch_y.size(0)
            correct += (predicted == batch_y).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

# =================== 阶段1：预训练自编码器 ===================

print("\n" + "="*60)
print("阶段1：自监督预训练 AutoEncoder")
print("="*60)

ae_model = EEGAutoEncoder(channels=62, samples=400, latent_dim=64).to(DEVICE)
ae_optimizer = optim.Adam(ae_model.parameters(), lr=LR)

print(f"预训练 {PRETRAIN_EPOCHS} 轮...")
for epoch in range(PRETRAIN_EPOCHS):
    loss = pretrain_epoch(ae_model, pretrain_loader, ae_optimizer, DEVICE)
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}/{PRETRAIN_EPOCHS} | Recon Loss: {loss:.6f}")

print("预训练完成！")

# =================== 阶段2：分类微调 ===================

print("\n" + "="*60)
print("阶段2：冻结编码器，训练分类头")
print("="*60)

classifier = EEGClassifier(ae_model.encoder, num_classes=3, latent_dim=64).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(classifier.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8, verbose=True)

best_val_acc = 0
best_epoch = 0
no_improve = 0

print(f"训练 {FINETUNE_EPOCHS} 轮...")
print("-" * 60)

for epoch in range(FINETUNE_EPOCHS):
    train_loss, train_acc = train_clf_epoch(classifier, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_preds, val_labels = eval_clf_epoch(classifier, val_loader, criterion, DEVICE)
    scheduler.step(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{FINETUNE_EPOCHS} | Train: {train_acc:.4f} | Val: {val_acc:.4f}")
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_model_state = classifier.state_dict().copy()
        no_improve = 0
    else:
        no_improve += 1
        
    if no_improve >= PATIENCE:
        print(f"\n⏹️ Early stopping (best={best_val_acc:.4f} at epoch {best_epoch})")
        break

classifier.load_state_dict(best_model_state)

# =================== 最终评估 ===================

print(f"\n[评估] 验证集 (Epoch {best_epoch})...")
print("-" * 60)
_, _, val_preds, val_labels = eval_clf_epoch(classifier, val_loader, criterion, DEVICE)
print(f"验证集准确率: {accuracy_score(val_labels, val_preds):.4f}")
print(classification_report(val_labels, val_preds, target_names=['negative', 'neutral', 'positive']))
print("混淆矩阵:")
print(confusion_matrix(val_labels, val_preds))

# =================== 预测 ===================

print("\n[预测] 生成测试集结果...")
classifier.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(DEVICE)
    test_outputs = classifier(X_test_device)
    _, test_pred = torch.max(test_outputs, 1)
    test_pred = test_pred.cpu().numpy()

output_file = 'seed_pretrain_predictions.txt'
np.savetxt(output_file, test_pred, fmt='%d')
print(f"\n✅ 已保存: {output_file}")
print(f"   negative={np.sum(test_pred==0)}, neutral={np.sum(test_pred==1)}, positive={np.sum(test_pred==2)}")

print("\n" + "=" * 60)
print("完成！")
print("=" * 60)

使用设备: cuda
[步骤1] 加载数据...
  预训练数据: (1350, 62, 400)
  分类训练集: (1147, 62, 400)
  分类验证集: (203, 62, 400)

阶段1：自监督预训练 AutoEncoder
预训练 80 轮...
  Epoch 10/80 | Recon Loss: 23.989321
  Epoch 20/80 | Recon Loss: 23.465388
  Epoch 30/80 | Recon Loss: 22.780416
  Epoch 40/80 | Recon Loss: 21.972832
  Epoch 50/80 | Recon Loss: 21.595541
  Epoch 60/80 | Recon Loss: 21.509504
  Epoch 70/80 | Recon Loss: 20.801348
  Epoch 80/80 | Recon Loss: 20.369073
预训练完成！

阶段2：冻结编码器，训练分类头
训练 100 轮...
------------------------------------------------------------
Epoch   1/100 | Train: 0.3636 | Val: 0.3251
Epoch   5/100 | Train: 0.3566 | Val: 0.3300
Epoch  10/100 | Train: 0.3409 | Val: 0.3300
Epoch 00012: reducing learning rate of group 0 to 5.0000e-04.
Epoch  15/100 | Train: 0.3514 | Val: 0.3300
Epoch  20/100 | Train: 0.3278 | Val: 0.3300
Epoch 00021: reducing learning rate of group 0 to 2.5000e-04.

⏹️ Early stopping (best=0.3350 at epoch 3)

[评估] 验证集 (Epoch 3)...
-----------------------------------------------------